# Step 51 — Composition counts in reporting windows

All step-specific code is below. It reads the trusted local catalogue pickle created by step 50 and writes two daily CSVs. It trusts the seven `event_class:` labels, including P_H, rather than inferring scientific content from temporary export headers. Unknown labels and records with no usable representative time are skipped and counted in the console summary.

## Reporting windows

The windows are derived from the UTC catalogue, not typed in. A day counts as covered when it has at least one event with a waveform record (W), one with picks (P) and one with a hypocentre (H). Covered days separated by at most `max_gap_days` (3) uncovered days form one window. Partial first and last days are then trimmed: an edge day is dropped while any source has fewer events than `min_edge_fraction` (0.25) times that window's daily median.

On the UTC catalogue this gives 8 May–10 June and 29 June–18 August 1991. The earlier hand-picked windows (8 May–9 June, 30 June–18 August) were chosen on recorded times; the UTC shift moves hypocentres into 10 June and picks and hypocentres into 29 June. Days inside a window that lack a source (for example 6 July, with no monthly picks) are listed in the summary and kept.

These are windows in which all three sources were being produced, **not** demonstrated complete detection or complete pick coverage. The legacy filename says “complete”; do not repeat that claim without an independent completeness analysis. Pass `windows=[(start, end), ...]` to use explicit dates instead.

`51_daily_source_coverage.csv` lists W, P and H counts for every day, with `all_sources` and `in_window` flags, so the windows can be checked. `51_daily_event_counts.csv` has one row for every day inside the windows (zero rows included).

Representative time priority: preferred origin, first timed origin, waveform-start comment, then earliest timed pick. Therefore this daily count may differ from step-33 waveform-start counts or step-53 S-file-filename dates. UTC representation does not establish the source clocks' absolute accuracy. Source-only classes are legitimate archive records and are retained.

Class columns partition the selected catalogue. An all-record catalogue and its waveform subset overlap; do not add their totals. Rebuilding the CSVs never changes the catalogue.

## Configuration

Inspect all input/output paths and settings. This cell resolves configuration; definitions and run cells follow.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'lib' / 'clock.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from lib.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    catalog_pkl=P['CATALOG_PREFIX'] + '.pkl',
    out_daily_csv=str(Path(P['TMP_DIR']) / '51_daily_event_counts.csv'),
    out_coverage_csv=str(Path(P['TMP_DIR']) / '51_daily_source_coverage.csv'),
    windows=None,           # None: derive from the catalogue; or [('1991-05-08', '1991-06-09'), ...]
    max_gap_days=3,         # uncovered days allowed inside one window
    min_edge_fraction=0.25, # trim edge days below this fraction of the window median
)
display(SETTINGS)

Configuration: /Users/thompsong/Developer/Pinatubo1991SeismicData/10_conversion/conversion_config.json
External outputs: /Users/thompsong/work/Pinatubo1991Processing/utc


{'catalog_pkl': '/Users/thompsong/work/Pinatubo1991Processing/utc/FAIR/metadata/50_pin_catalog.pkl',
 'out_daily_csv': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/51_daily_event_counts.csv',
 'out_coverage_csv': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/51_daily_source_coverage.csv',
 'windows': None,
 'max_gap_days': 3,
 'min_edge_fraction': 0.25}

## Workflow implementation

The next cell defines functions only. It does not execute the workflow or write outputs.

In [2]:
from __future__ import annotations

import pickle
from pathlib import Path
from typing import Optional

import pandas as pd
from obspy import UTCDateTime
from obspy.core.event import Event

from flovopy.core.event_utils import get_comment_value

EXPECTED_LABELS = ["W_P_H", "W_P", "W_H", "W_ONLY", "P_H", "P_ONLY", "H_ONLY"]
SOURCES = ("W", "P", "H")  # waveform record, picks, hypocentre


def get_event_class(event: Event) -> str:
    """Read event_class from step 50; UNKNOWN if missing."""
    val = get_comment_value(event, "event_class:")
    return "UNKNOWN" if val is None else val.strip().upper()


def class_sources(label: str) -> set:
    """Sources present in an event class: 'W_P' -> {'W', 'P'}, 'H_ONLY' -> {'H'}."""
    return set(label.replace("_ONLY", "").split("_")) & set(SOURCES)


def infer_event_time(event: Event) -> Optional[UTCDateTime]:
    """Representative time: preferred origin, first origin, waveform start, earliest pick."""
    o = event.preferred_origin()
    if o and o.time:
        return o.time
    if event.origins and event.origins[0].time:
        return event.origins[0].time
    wst = get_comment_value(event, "waveform_starttime:")
    if wst:
        try:
            return UTCDateTime(wst)
        except Exception:
            pass
    times = [p.time for p in event.picks if p.time]
    return min(times) if times else None


def classify_events(catalog):
    """One row per event with a time and a known class, plus counts of skipped events."""
    rows, skipped = [], {"no_time": 0, "unknown_class": 0}
    for ev in catalog:
        t = infer_event_time(ev)
        if t is None:
            skipped["no_time"] += 1
            continue
        label = get_event_class(ev)
        if label not in EXPECTED_LABELS:
            skipped["unknown_class"] += 1
            continue
        rows.append({"date": pd.Timestamp(t.datetime).normalize(),
                     "event_time_utc": t.datetime, "event_class": label})
    return pd.DataFrame(rows, columns=["date", "event_time_utc", "event_class"]), skipped


def daily_source_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Events per UTC day that include each source (W, P, H), on a continuous date range."""
    flags = pd.DataFrame({s: df["event_class"].map(lambda c: s in class_sources(c)) for s in SOURCES})
    day = flags.groupby(df["date"]).sum().astype(int)
    full = pd.date_range(day.index.min(), day.index.max(), freq="D", name="date")
    return day.reindex(full, fill_value=0)


def derive_coverage_windows(day: pd.DataFrame, *, max_gap_days: int = 3,
                            min_edge_fraction: float = 0.25):
    """Date ranges in which waveforms, picks and hypocentres all exist.

    1. A day is covered when every source has at least one event.
    2. Covered days separated by at most ``max_gap_days`` uncovered days form one window.
    3. Partial first and last days are trimmed: an edge day is dropped while any source
       has fewer events than ``min_edge_fraction`` times that window's median.
    Returns a list of (start, end) ISO date strings, both inclusive.
    """
    covered = day.index[(day[list(SOURCES)] > 0).all(axis=1)]
    runs = []
    for d in covered:
        if runs and (d - runs[-1][1]).days <= max_gap_days + 1:
            runs[-1][1] = d
        else:
            runs.append([d, d])
    windows = []
    for a, b in runs:
        w = day.loc[a:b, list(SOURCES)]
        floor = min_edge_fraction * w.median()
        partial = (w < floor).any(axis=1)
        keep = w.index[~partial]
        if len(keep):
            windows.append((keep.min().strftime("%Y-%m-%d"), keep.max().strftime("%Y-%m-%d")))
    return windows


def in_windows(dates: pd.Series, windows) -> pd.Series:
    """Whether each UTC date (normalized timestamp) lies in any inclusive window."""
    mask = pd.Series(False, index=dates.index)
    for a, b in windows:
        mask |= dates.between(pd.Timestamp(a), pd.Timestamp(b))
    return mask


def run(*, catalog_pkl, out_daily_csv, out_coverage_csv, windows=None,
        max_gap_days=3, min_edge_fraction=0.25):
    """Count event classes per UTC day inside the reporting windows.

    ``windows=None`` derives the windows from the catalogue (see
    ``derive_coverage_windows``); otherwise give (start, end) date pairs.
    """
    with open(catalog_pkl, "rb") as f:
        catalog = pickle.load(f)
    df, skipped = classify_events(catalog)
    if df.empty:
        print("No events with a time and a known class.")
        return {"df": df, "skipped": skipped}

    day = daily_source_counts(df)
    if windows is None:
        windows = derive_coverage_windows(day, max_gap_days=max_gap_days,
                                          min_edge_fraction=min_edge_fraction)
        window_source = f"derived (max_gap_days={max_gap_days}, min_edge_fraction={min_edge_fraction})"
    else:
        windows = [tuple(w) for w in windows]
        window_source = "explicit"

    coverage = day.reset_index()
    coverage["all_sources"] = (coverage[list(SOURCES)] > 0).all(axis=1)
    coverage["in_window"] = in_windows(coverage["date"], windows)
    out_cov = Path(out_coverage_csv)
    out_cov.parent.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(out_cov, index=False, date_format="%Y-%m-%d")

    df_in = df[in_windows(df["date"], windows)]
    daily = (df_in.pivot_table(index="date", columns="event_class", values="event_time_utc",
                               aggfunc="count", fill_value=0)
             .reindex(columns=EXPECTED_LABELS, fill_value=0))
    # Every day of every window gets a row, including days with no events.
    all_days = pd.DatetimeIndex(sorted({d for a, b in windows
                                        for d in pd.date_range(a, b, freq="D")}), name="date")
    daily = daily.reindex(all_days, fill_value=0).reset_index()
    daily["Total"] = daily[EXPECTED_LABELS].sum(axis=1)
    out_csv = Path(out_daily_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    daily.to_csv(out_csv, index=False, date_format="%Y-%m-%d")

    gaps = coverage[coverage["in_window"] & ~coverage["all_sources"]]
    comp = df_in["event_class"].value_counts()
    print("EVENT-CLASS COUNTS IN REPORTING WINDOWS")
    print("Windows (UTC, inclusive):", window_source)
    for a, b in windows:
        print(f"  - {a} .. {b}")
    if len(gaps):
        print("Days inside a window missing a source:")
        for _, r in gaps.iterrows():
            missing = [s for s in SOURCES if r[s] == 0]
            print(f"  - {r['date']:%Y-%m-%d}: no {', '.join(missing)}")
    print(f"\nTotal events in windows: {len(df_in)}")
    for k in EXPECTED_LABELS:
        print(f"{k:8s}: {comp.get(k, 0)}")
    print(f"\nDaily counts:    {out_csv}")
    print(f"Daily coverage:  {out_cov}")
    if skipped["no_time"]:
        print(f"Skipped events with no time: {skipped['no_time']}")
    if skipped["unknown_class"]:
        print(f"Skipped events with an unknown event_class: {skipped['unknown_class']}")
    return {"df": df, "df_daily": daily, "df_coverage": coverage,
            "windows": windows, "skipped": skipped}


## Run and inspect

Run this cell only after the configuration and all definitions. It writes the configured outputs; inspect the summary and exceptions before proceeding.

In [3]:
result = run(**SETTINGS)

EVENT-CLASS COUNTS IN REPORTING WINDOWS
Windows (UTC, inclusive): derived (max_gap_days=3, min_edge_fraction=0.25)
  - 1991-05-08 .. 1991-06-10
  - 1991-06-29 .. 1991-08-18
Days inside a window missing a source:
  - 1991-05-10: no H
  - 1991-05-12: no H
  - 1991-05-18: no H
  - 1991-05-19: no H
  - 1991-05-28: no H
  - 1991-05-29: no H
  - 1991-07-06: no P, H

Total events in windows: 11753
W_P_H   : 4343
W_P     : 2698
W_H     : 145
W_ONLY  : 4288
P_H     : 121
P_ONLY  : 97
H_ONLY  : 61

Daily counts:    /Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/51_daily_event_counts.csv
Daily coverage:  /Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/51_daily_source_coverage.csv


In [4]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)

df


,date,event_time_utc,event_class
0,1991-07-31,1991-07-31 16:06:14.810,P_ONLY
1,1991-07-31,1991-07-31 16:07:16.650,P_H
2,1991-07-31,1991-07-31 16:21:28.750,P_ONLY
3,1991-07-31,1991-07-31 16:28:32.110,P_H
4,1991-07-31,1991-07-31 16:36:06.910,P_H


df_daily


event_class,date,W_P_H,W_P,W_H,W_ONLY,P_H,P_ONLY,H_ONLY,Total
0,1991-05-08,4,76,0,7,0,0,0,87
1,1991-05-09,8,38,0,7,0,0,0,53
2,1991-05-10,0,1,0,2,0,0,0,3
3,1991-05-11,6,28,0,3,0,0,0,37
4,1991-05-12,0,35,0,6,0,0,0,41


df_coverage


,date,W,P,H,all_sources,in_window
0,1991-04-30,9,9,0,False,False
1,1991-05-01,15,15,0,False,False
2,1991-05-02,27,24,0,False,False
3,1991-05-03,55,52,0,False,False
4,1991-05-04,63,47,0,False,False


windows


[('1991-05-08', '1991-06-10'), ('1991-06-29', '1991-08-18')]

skipped


{'no_time': 0, 'unknown_class': 0}